In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
dataset_filepath = "application_resource_average_usage_data.csv"

In [ ]:
df = pd.read_csv(dataset_filepath)
df.head()

In [ ]:
# check null values
df.isnull().sum()

In [ ]:
# duplicates
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df["Application_Category"].nunique()

In [ ]:
sns.countplot(x="Application_Category", data=df)
plt.title("Distribution of Application Categories")
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data=df, x=col, hue="Application_Category", fill=True)
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()


In [ ]:
df_encoded = df.copy()
le = LabelEncoder()
df_encoded["App_Label"] = le.fit_transform(df_encoded["Application_Category"])

# Correlation matrix
corr = df_encoded.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# Top features most correlated with App_Label
corr_target = corr["App_Label"].drop("App_Label").sort_values(ascending=False)
print("Top positively correlated features:\n", corr_target.head())
print("\nTop negatively correlated features:\n", corr_target.tail())


In [ ]:
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x="Application_Category", y=col, data=df)
    plt.title(f"{col} by Application Category")
    plt.tight_layout()
    plt.show()


In [ ]:
sns.pairplot(df_encoded, hue="Application_Category", 
             vars=["CPU_Usage(%)", "Memory_Usage(MB)", "Requests_per_sec", "Error_Rate(%)"], plot_kws={"s": 5})
plt.suptitle("Pairwise Feature Relationships", y=1.02)
plt.show()


In [ ]:
X = df_encoded.drop(columns=["Application_Category", "App_Label"])
y = df_encoded["App_Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,test_size=0.2, random_state=42)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

importances = pd.Series(dt_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=True).plot(kind='barh', figsize=(8, 6), title="Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

svm_model = make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42))
svm_model.fit(X_train, y_train)


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
mlp_model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, X_test, y_test, name):
    print(f"\n===== {name} Evaluation =====")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

evaluate_model(dt_model, X_test, y_test, "Decision Tree")
evaluate_model(svm_model, X_test, y_test, "SVM")
evaluate_model(mlp_model, X_test, y_test, "Neural Network")


In [ ]:
import joblib

# Save the best model — choose based on accuracy/F1
joblib.dump(svm_model, "app_perf_classifier_model.joblib")
joblib.dump(le, "app_classify_label_encoder.joblib")  # save label encoder too


In [1]:
import joblib

# Load the saved model and label encoder
model = joblib.load("app_perf_classifier_model.joblib")
label_encoder = joblib.load("app_classify_label_encoder.joblib")


In [3]:
new_data = {
    "CPU_Usage(%)": 45,
    "Memory_Usage(MB)": 1800,
    "Disk_IO(MB/s)": 60,
    "Network_Usage(Mbps)": 90,
    "GPU_Utilization(%)": 25,
    "Execution_Time(sec)": 4,
    "Response_Time(ms)": 250,
    "Requests_per_sec": 180,
    "Error_Rate(%)": 1.2,
    "Crash_Frequency": 0.5,
    "Memory_Leak_Rate(MB/sec)": 0.4,
    "User_Load": 320,
    "Peak_Idle_Ratio": 0.35
}

import pandas as pd

new_df = pd.DataFrame([new_data])


# Predict numerical class
predicted_class = model.predict(new_df)[0]

# Convert to readable label
predicted_label = label_encoder.inverse_transform([predicted_class])[0]

print("Predicted Application Category:", predicted_label)


Predicted Application Category: Efficient


In [6]:
proba = model.predict_proba(new_df)[0]
for cls, p in zip(label_encoder.classes_, proba):
    print(f"{cls}: {p:.2f}")


Efficient: 0.62
Moderate: 0.38
Resource-Intensive: 0.00


In [7]:
print(proba)

[6.18725817e-01 3.81215350e-01 5.88329324e-05]


In [9]:
# Define class score mapping
score_mapping = {
    "Efficient": 0.0,
    "Moderate": 0.5,
    "Resource-Intensive": 1.0
}

# Get probabilities for each class
proba = model.predict_proba(new_df)[0]

# Map labels to numeric scores
class_labels = label_encoder.classes_
class_scores = [score_mapping[label] for label in class_labels]

# Calculate weighted score
final_score = sum(p * s for p, s in zip(proba, class_scores))

# Predicted label (for display)
predicted_label = label_encoder.inverse_transform([model.predict(new_df)[0]])[0]

# Output
print(f"Predicted Category: {predicted_label}")
print(f"Weighted Performance Score: {final_score:.2f}")


Predicted Category: Efficient
Weighted Performance Score: 0.19
